# 03 — Kelly bet-sizing sweep

How does the bankroll grow under different Kelly configurations? Two knobs:

- **Kelly fraction**: multiplier on the full Kelly stake. `1.0` = full Kelly (aggressive). `0.25` = quarter-Kelly (PLAN.md §10.2 default).
- **Max-bet cap**: hard cap on per-bet stake as a fraction of bankroll, regardless of what Kelly says. PLAN.md §10.2 default is 2%.

The PLAN's safety net (¼-Kelly + 2% cap) is conservative for our models — this notebook quantifies the tradeoff.

**Scenario**: Kalshi-like (no-vig prices + 7% fee on winnings).
**Edge threshold**: 3% (PLAN.md §10.1).
**Starting bankroll**: $1. Final bankroll is the growth factor.
**Period**: validation 2023 (504 fights, ~360 bets at edge ≥ 3%).

**Three models compared:**
1. **v3_full2000** — the current champion. Single CatBoost on v3 scalar Bayesian skill features, no early stopping.
2. **v7** — the conservative deploy alternative. 10-seed ensemble of v3.3, better calibrated for Kelly.
3. **v3_full2000_no_skill_corrupted** — v3_full2000 wrapped so skill columns are forced to NaN at inference. Reproduces an earlier diagnostic-bug behavior that produced surprisingly good no-cap Kelly numbers. Preserved as a deployable model so test-set evaluation can decide whether those numbers were signal or luck.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from ufc_pred.backtest.bet_eval import evaluate_bets_kelly
from ufc_pred.features.static_v1 import prepare
from ufc_pred.features.skill_v3_pipeline import OUTPUT as SKILL_V3_PARQUET
from ufc_pred.features.skill_v3_1_pipeline import OUTPUT as SKILL_V3_1_PARQUET
from ufc_pred.ingest.kaggle_mdabbert import HISTORY_PARQUET
from ufc_pred.utils.time_splits import split

pd.set_option('display.precision', 3)

## Data + predictions

Load val fights, join both skill feature versions (so all v3.x and v7 variants can be served from one frame),
and generate val predictions for the two models we care about.

In [ ]:
fights = pd.read_parquet(HISTORY_PARQUET)
fights = fights[fights['Winner'].isin(['Red', 'Blue'])].copy()
fights['date'] = pd.to_datetime(fights['date'])

# Join v3 skill (used by v3_full2000 and the corrupted wrapper)
sk_v3 = pd.read_parquet(SKILL_V3_PARQUET)
sk_v3['date'] = pd.to_datetime(sk_v3['date'])
fights = fights.merge(
    sk_v3[['date', 'R_fighter', 'B_fighter', 'skill_diff_mean', 'skill_diff_std']],
    on=['date', 'R_fighter', 'B_fighter'], how='left', validate='many_to_one',
)

# Also join v3 + v3.1 under suffixed names (used by v7)
sk_v3_s = sk_v3.rename(columns={'skill_diff_mean': 'skill_diff_mean_v3',
                                 'skill_diff_std': 'skill_diff_std_v3'})
sk_v3_1 = pd.read_parquet(SKILL_V3_1_PARQUET).rename(columns={
    'skill_diff_mean': 'skill_diff_mean_v3_1',
    'skill_diff_std': 'skill_diff_std_v3_1'})
sk_v3_1['date'] = pd.to_datetime(sk_v3_1['date'])
fights = fights.merge(
    sk_v3_s[['date','R_fighter','B_fighter','skill_diff_mean_v3','skill_diff_std_v3']],
    on=['date','R_fighter','B_fighter'], how='left', validate='many_to_one')
fights = fights.merge(
    sk_v3_1[['date','R_fighter','B_fighter','skill_diff_mean_v3_1','skill_diff_std_v3_1']],
    on=['date','R_fighter','B_fighter'], how='left', validate='many_to_one')

splits = split(fights)
val = splits.val.reset_index(drop=True)
y_val = (val['Winner'].to_numpy() == 'Red').astype(int)

def predict(model_name):
    payload = joblib.load(ROOT / 'artifacts' / 'models' / f'{model_name}.joblib')
    cols = payload['columns']
    cat = payload.get('cat_features', [])
    X, _, _, _ = prepare(val, augment_symmetry=False, one_hot=False)
    X = X.reindex(columns=cols, fill_value=None)
    for c in cat:
        X[c] = X[c].fillna('__missing__').astype(str)
    if 'models' in payload:        # ensemble (v7, v7.1)
        return np.mean([m.predict_proba(X)[:, 1] for m in payload['models']], axis=0)
    return payload['model'].predict_proba(X)[:, 1]  # single (v3_full2000) or CorruptedSkillModel wrapper

p_v3 = predict('v3_catboost_full2000')
p_v7 = predict('v7_catboost_ensemble')
p_corrupt = predict('v3_full2000_no_skill_corrupted')

print(f"v3_full2000:           pred mean {p_v3.mean():.3f}, range [{p_v3.min():.3f}, {p_v3.max():.3f}]")
print(f"v7 ensemble:           pred mean {p_v7.mean():.3f}, range [{p_v7.min():.3f}, {p_v7.max():.3f}]")
print(f"v3_full2000_corrupted: pred mean {p_corrupt.mean():.3f}, range [{p_corrupt.min():.3f}, {p_corrupt.max():.3f}]")
print(f"val n: {len(val)}  y_red mean: {y_val.mean():.3f}")

## Sweep grid

For each (model, Kelly fraction, max-bet cap) tuple, simulate the bankroll over val 2023 and record the final
balance, total return, max drawdown, and full trajectory. `1.0` in the cap column means *no cap* — Kelly
gets to stake whatever its formula dictates, subject only to the fraction multiplier.

In [ ]:
FRACTIONS = [0.10, 0.25, 0.50, 1.00]
CAPS = [0.01, 0.02, 0.05, 0.10, 1.00]   # 1.00 effectively = no cap
EDGE_THRESHOLD = 0.03
FEE_RATE = 0.07
USE_NO_VIG = True
START_BANKROLL = 1.0

def sweep(p_val):
    """Run the Kelly grid for one model. Returns dict[(frac, cap)] → result dict."""
    out = {}
    for f in FRACTIONS:
        for c in CAPS:
            r = evaluate_bets_kelly(
                p_val, y_val, val['R_odds'], val['B_odds'],
                edge_threshold=EDGE_THRESHOLD, fee_rate=FEE_RATE, use_no_vig=USE_NO_VIG,
                kelly_fraction=f, max_bet_fraction=c, starting_bankroll=START_BANKROLL,
            )
            out[(f, c)] = r
    return out

grid_v3 = sweep(p_v3)
grid_v7 = sweep(p_v7)
grid_corrupt = sweep(p_corrupt)

print(f"Sweep done: {len(grid_v3)} configs per model × 3 models")
print(f"Trajectory length: {len(grid_v3[(0.25, 0.02)]['trajectory'])} bankroll points")

## Final-bankroll heatmaps

Each cell is the value of a $1 starting bankroll after the entire val sequence of bets. Rows = Kelly fraction
(more aggressive lower); columns = per-bet cap (more permissive right).

Read the gradient: green = bankroll grew, white ≈ break-even, red = bankroll shrank. The bottom-right cell
(full Kelly, no cap) is where the math says "stake everything" — and is also where ruin lives.

In [ ]:
def grid_to_df(grid, key):
    """Pivot a (frac, cap) → result dict into a DataFrame of one metric."""
    data = {c: [grid[(f, c)][key] for f in FRACTIONS] for c in CAPS}
    df = pd.DataFrame(data, index=[f'{int(f*100)}%-K' for f in FRACTIONS])
    df.columns = [(f'{int(c*100)}%' if c < 1 else 'no cap') for c in CAPS]
    df.columns.name = 'per-bet cap'
    df.index.name = 'Kelly fraction'
    return df

bk_v3 = grid_to_df(grid_v3, 'final_bankroll')
bk_v7 = grid_to_df(grid_v7, 'final_bankroll')
bk_corrupt = grid_to_df(grid_corrupt, 'final_bankroll')

print("CHAMPION — v3_full2000 final bankroll ($1 → ?)")
display(bk_v3.style
    .format('${:.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=3.0, axis=None))

print("\nBACKUP — v7 ensemble final bankroll ($1 → ?)")
display(bk_v7.style
    .format('${:.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=3.0, axis=None))

print("\nCORRUPTED — v3_full2000_no_skill final bankroll ($1 → ?)")
display(bk_corrupt.style
    .format('${:.2f}')
    .background_gradient(cmap='RdYlGn', vmin=0.5, vmax=3.0, axis=None))

## Max-drawdown heatmaps

Drawdown is the worst peak-to-trough drop the bankroll experienced *during* the simulation. PLAN.md §10.3 says
step down to 1/8 Kelly once you hit −25%, and pause entirely on persistent losses. So any cell ≥ 25% is in
"step-down territory" — the model would have triggered the rule at least once during val.

In [ ]:
dd_v3 = grid_to_df(grid_v3, 'max_drawdown_pct')
dd_v7 = grid_to_df(grid_v7, 'max_drawdown_pct')
dd_corrupt = grid_to_df(grid_corrupt, 'max_drawdown_pct')

print("CHAMPION — v3_full2000 max drawdown (%)")
display(dd_v3.style
    .format('{:.1f}%')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=80, axis=None))

print("\nBACKUP — v7 ensemble max drawdown (%)")
display(dd_v7.style
    .format('{:.1f}%')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=80, axis=None))

print("\nCORRUPTED — v3_full2000_no_skill max drawdown (%)")
display(dd_corrupt.style
    .format('{:.1f}%')
    .background_gradient(cmap='RdYlGn_r', vmin=0, vmax=80, axis=None))

## Bankroll over time — all 20 configurations per model

The line graph asked for. Each line is one (Kelly fraction, cap) tuple. The x-axis is bet number in chronological
order through val 2023; the y-axis is the bankroll value (started at $1).

Two presentations side by side:

- **Linear scale (left)**: easy to read drawdowns and small-bankroll regimes; large bankrolls clip at the top.
- **Log scale (right)**: shows the full dynamic range from $0.01 (effective ruin) to $100+. Compound growth
  appears as straight lines; the gap between configurations becomes the per-bet edge.

Color encodes the Kelly fraction (darker = more aggressive). Line style encodes the cap.

In [ ]:
FRACTION_COLORS = {0.10: '#a6d96a', 0.25: '#1a9641', 0.50: '#fdae61', 1.00: '#d7191c'}
CAP_LINESTYLES = {0.01: ':', 0.02: '--', 0.05: '-.', 0.10: '-', 1.00: (0, (3, 1, 1, 1))}
CAP_LABELS = {0.01: '1%', 0.02: '2%', 0.05: '5%', 0.10: '10%', 1.00: 'no cap'}


def plot_trajectories(grid, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, scale in zip(axes, ['linear', 'log']):
        for f in FRACTIONS:
            for c in CAPS:
                traj = grid[(f, c)]['trajectory']
                final = grid[(f, c)]['final_bankroll']
                label = f"{int(f*100)}%-K, cap {CAP_LABELS[c]} → ${final:.2f}"
                ax.plot(traj, color=FRACTION_COLORS[f], linestyle=CAP_LINESTYLES[c],
                        linewidth=1.6, alpha=0.9, label=label)
        ax.axhline(1.0, color='black', linestyle='-', alpha=0.4, linewidth=0.8)
        ax.axhline(0.75, color='red', linestyle=':', alpha=0.4, linewidth=0.8)  # PLAN.md −25% step-down
        ax.set_xlabel('Bet # (chronological order through val 2023)')
        ax.set_ylabel('Bankroll (starting $1)')
        ax.set_yscale(scale)
        if scale == 'log':
            ax.set_ylim(0.01, 200)
        ax.set_title(f"{title} — {scale} scale")
        ax.grid(alpha=0.3)
    # One shared legend outside both axes
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=7, framealpha=0.95, ncol=1)
    plt.tight_layout()
    plt.show()


plot_trajectories(grid_v3, "CHAMPION  v3_full2000")
print("Notes: dashed black = $1 break-even.  Dotted red = $0.75 PLAN.md −25% step-down line.")
print("Color = Kelly fraction (green=10% / dark-green=25% / orange=50% / red=100%).")
print("Linestyle = per-bet cap (dotted=1% / dashed=2% / dash-dot=5% / solid=10% / loose-dash=no cap).")

In [ ]:
plot_trajectories(grid_v7, "BACKUP  v7 ensemble (better calibrated)")

In [ ]:
plot_trajectories(grid_corrupt, "CORRUPTED  v3_full2000 + NaN skill features")

## Side-by-side: champion vs backup at the most-likely deploy profile

The PLAN.md default (¼-Kelly, 2% cap) versus the suggested middle-ground (¼-Kelly, 5% cap) — for both models.
Tight focus on the four configurations most likely to be deployed in practice.

In [ ]:
DEPLOY_CONFIGS = [
    ('v3_full2000',         grid_v3,      0.25, 0.02, '#1a9641', '--', '¼-K, 2% (PLAN.md)'),
    ('v3_full2000',         grid_v3,      0.25, 0.05, '#1a9641', '-',  '¼-K, 5% (balanced)'),
    ('v7',                  grid_v7,      0.25, 0.02, '#d7191c', '--', '¼-K, 2% (PLAN.md)'),
    ('v7',                  grid_v7,      0.25, 0.05, '#d7191c', '-',  '¼-K, 5% (balanced)'),
    ('v3_full2000_corrupt', grid_corrupt, 0.25, 0.02, '#7570b3', '--', '¼-K, 2% (PLAN.md)'),
    ('v3_full2000_corrupt', grid_corrupt, 0.25, 0.05, '#7570b3', '-',  '¼-K, 5% (balanced)'),
]

fig, ax = plt.subplots(figsize=(11, 6))
for model, grid, f, c, color, ls, profile in DEPLOY_CONFIGS:
    r = grid[(f, c)]
    label = f"{model} | {profile} → ${r['final_bankroll']:.2f}  (DD {r['max_drawdown_pct']:.1f}%)"
    ax.plot(r['trajectory'], color=color, linestyle=ls, linewidth=2.5, alpha=0.95, label=label)
ax.axhline(1.0, color='black', linestyle='-', alpha=0.4, linewidth=0.8)
ax.axhline(0.75, color='red', linestyle=':', alpha=0.4, linewidth=0.8)
ax.set_xlabel('Bet # (chronological order through val 2023)')
ax.set_ylabel('Bankroll (starting $1)')
ax.set_title('Deploy candidates — quarter-Kelly at 2% (PLAN.md) vs 5% (balanced)')
ax.legend(loc='upper left', fontsize=9, framealpha=0.95)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Takeaways

What this sweep tells us — read with the caveat that we have only one val window (504 fights, ~360 bets) and
the underlying flat-stake ROI has a wide bootstrap CI that includes zero.

1. **PLAN.md's 2% cap is the binding constraint, not the Kelly fraction.** Moving cap from 2% to 5% with quarter-Kelly more than doubles the final bankroll for v3_full2000. The Kelly fraction has a much smaller effect once the cap is set.

2. **Full Kelly with no cap → ruin, even on a profitable model.** v3_full2000 reaches $0.00 in that cell despite having genuine positive edge. This is the classic "Kelly is dangerous without a safety net" failure mode playing out concretely. The corrupted variant survives full-Kelly + no cap better (it's less confident → smaller stakes), but at moderate caps it loses to the real model.

3. **Champion vs backup vs corrupted at the recommended deploy profile (¼-K + 2% cap):**
   - v3_full2000:           $1.90  (DD 25.5%)
   - v7:                    $1.36  (DD 26.2%)
   - v3_full2000_corrupted: $1.76  (DD 24.2%)
   
   At PLAN.md-default sizing, the real champion is still best. The corrupted version is competitive but not better.

4. **Where the corrupted model "wins" — and what it really means.** At permissive configurations (≥5% cap, ½-Kelly or full Kelly, and especially no-cap), the corrupted model's lower-confidence predictions translate into smaller individual stakes, which lets it survive ruinous configurations where the real model dies. Most extreme case: at ¼-K + no cap, corrupted reaches **$103.13** while the real champion only manages **$45.66**. This is not "the corrupted model is better" — it's "the corrupted model's coincidentally smaller stakes happened to dodge the wipeout that hits aggressive Kelly on a small val window." See finding #17 in STATUS.md for the full reasoning; we verified temperature-shrinking the real model does NOT reproduce this behavior, so it's not a calibration story — it's a "different bet selection on a specific val sequence" story.

5. **The 5% cap is in step-down territory for all three models.** Max-drawdowns at ¼-K + 5% are ~49% (v3_full2000), ~53% (v7), ~49% (corrupted) — all well past PLAN.md §10.3's −25% step-down threshold. Even the conservative 2% cap touches ~24-26% drawdown across the board. Whatever profile we deploy with, expect to hit step-down once or twice in a 12-month window.

6. **Recommended deploy candidates** (rank-ordered by expected outcome at PLAN.md sizing):
   - **Very conservative** (PLAN.md default): v3_full2000, ¼-K, 2% cap. Val outcome ~$1.90, max DD ~25%. The honest answer.
   - **Balanced**: v3_full2000, ¼-K, 5% cap. Val outcome ~$4.29, max DD ~49%. Acceptable if you can stomach the drawdown.
   - **Backup with PLAN.md profile**: v7, ¼-K, 2% cap. Val outcome ~$1.36, max DD ~26%. Lower expected growth but better calibration if you don't trust v3_full2000's overconfident probabilities.

7. **Reality check on the val numbers.** These compound returns assume the underlying flat-stake edge is real. The bootstrap CI on flat-stake ROI was wide enough that a true ROI near 0 is plausible — in which case the no-cap rows go to zero. Treat the +329% balanced-profile headline as the *upper end* of a noisy estimate. Live paper trading (Phase 9) is the only way to resolve which row of this grid we actually live in. **Test-set evaluation will eventually include all three models so we can see whether the corrupted variant's strange properties hold up out-of-distribution.**